# Biohub · v11 — fine-tune the Trackastra linker on the competition GT

**What this notebook produces** (in its Output): `models/ctc_ft/` — Trackastra weights re-trained on *our own
DoG detections* with the sparse `.geff` ground truth, plus `feat_cache/` (re-usable features, one `.npz` per clip).
Attach this notebook's Output to the inference notebook (v12+); it prefers `ctc_ft` over the pretrained `ctc`.

**Inputs to attach**: competition data + `biohub-prep-bundle` Output (wheels + pretrained model).
Optionally the Output of a *previous run of this notebook* → its `feat_cache` is re-used and feature extraction is skipped.

**Settings**: Accelerator GPU T4 x2, Internet off. Runtime ≈ features 3–4 h (first run) + training ≈ 3–4 h + CV 20 min.

How the sparse GT is used: a detection matched to a GT cell (≤ 7 µm, like the metric) whose GT successor is also
detected gives one supervised *row* (the successor is the positive, every other next-frame detection is a true
negative). Unlabeled cells are masked out of the loss entirely. Hold-out clips are the same 12 clips as the v9 CV.


In [ ]:
# ------------------------------------------------------------------ 0. settings
import os, sys, glob, time, json, subprocess, importlib, shutil
os.environ["TQDM_DISABLE"] = "1"   # quiet progress bars in the log
from pathlib import Path

DETECT_CONFIG = {"adaptive_radius": {"a": 26.5, "rmin": 3.0, "rmax": 6.0}, "merge_min_dist_um": 6.0, "thresh_k": 0.5}
                                  # = FINAL_CONFIG "adapt_a26_merge" of v8 (+ the calibrated thresh_k). Must match inference.
N_HOLDOUT         = 12            # hold-out clips = same rule as the v9 CV (cv_zarrs[1::step][:12]) -> honest comparison
MAX_TRAIN_CLIPS   = None          # None = all remaining train clips (order interleaves embryos)
FEATURE_BUDGET_SEC= 3.0 * 3600    # stop extracting features after this; training uses whatever is cached
FEATURE_WORKERS   = 4             # joblib workers for regionprops feature extraction
TRAIN_STEPS       = 3000          # optimizer steps (each = ACCUM windows)
ACCUM             = 8
LR                = 5e-5
WARMUP_STEPS      = 200
AUGMENT_LEVEL     = 2             # 0 none | 1 flip+affine | 2 + brightness + jitter
MAX_TOKENS        = 3000          # windows with more detections drop trailing frames (dense clips: ~600/frame x 4)
TRAIN_BUDGET_SEC  = 5.5 * 3600    # features took ~1.9 h on Kaggle for all 199 clips
EVAL_EVERY        = 250
MODEL_PREFER      = "ctc"         # start fine-tuning from the pretrained weights (use "ctc_ft" to continue a previous fine-tune)
SEED              = 0

KAGGLE_INPUT = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))
OUT_DIR      = Path(os.environ.get("KAGGLE_OUTPUT_DIR", "/kaggle/working"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
T_START = time.time()
print("input root:", KAGGLE_INPUT, "| output:", OUT_DIR)


In [ ]:
# ------------------------------------------------------------------ 1. locate inputs (fast: no recursive glob over the 87 GB dataset)
COMP_SLUG = "biohub-cell-tracking-during-development"

def _find_comp_dir():
    for c in [KAGGLE_INPUT / "competitions" / COMP_SLUG, KAGGLE_INPUT / COMP_SLUG]:
        if (c / "test").is_dir():
            return c
    for c in KAGGLE_INPUT.glob("*/"):              # one level deep only
        if (c / "test").is_dir() and (c / "sample_submission.csv").exists():
            return c
    for c in KAGGLE_INPUT.glob("*/*/"):
        if (c / "test").is_dir() and (c / "sample_submission.csv").exists():
            return c
    return None

def _walk_pruned(root, max_depth=8):
    """os.walk that never descends into the competition data or zarr/geff stores."""
    root = str(root)
    for dp, dns, fns in os.walk(root):
        depth = dp[len(root):].count(os.sep)
        dns[:] = [d for d in dns if not d.endswith((".zarr", ".geff")) and d != COMP_SLUG and d != "competitions"]
        if depth >= max_depth:
            dns[:] = []
        yield Path(dp), fns

COMP_DIR = _find_comp_dir()
MODEL_DIRS, PKG_DIR, CACHE_DIRS, REPO_DIRS, CKPT_DIRS = [], None, [], [], []
for dp, fns in _walk_pruned(KAGGLE_INPUT):
    if "model.pt" in fns and "train_config.yaml" in fns:
        MODEL_DIRS.append(dp)
    if PKG_DIR is None and any(f.startswith("trackastra-") and f.endswith(".whl") for f in fns):
        PKG_DIR = dp
    if dp.name == "feat_cache" and any(f.endswith(".npz") for f in fns):
        CACHE_DIRS.append(dp)
    if dp.name == "tracking_cellmot" and "io.py" in fns:            # official baseline repo (prep-bundle v2: repo/src/tracking_cellmot)
        REPO_DIRS.append(dp.parent.parent)
    if "edge_predictor_best.pth" in fns or "last.pth" in fns:        # baseline checkpoints (ours or the repo's layout)
        CKPT_DIRS.append(dp)
# model folder preference: MODEL_PREFER (e.g. "ctc_ft" = fine-tuned) first, then the pretrained "ctc"
_pref = globals().get("MODEL_PREFER", "ctc_ft")
MODEL_DIRS.sort(key=lambda p: (p.name != _pref, p.name != "ctc", str(p)))
MODEL_DIR = MODEL_DIRS[0] if MODEL_DIRS else None
PRETRAINED_DIR = next((p for p in MODEL_DIRS if p.name == "ctc"), MODEL_DIR)

print("COMP_DIR :", COMP_DIR)
print("MODEL_DIR:", MODEL_DIR, "(all found:", [str(p) for p in MODEL_DIRS], ")")
print("PKG_DIR  :", PKG_DIR)
print("CACHE_DIRS:", [str(p) for p in CACHE_DIRS])
REPO_DIR = REPO_DIRS[0] if REPO_DIRS else None
print("REPO_DIR :", REPO_DIR, "| CKPT_DIRS:", [str(p) for p in CKPT_DIRS])
print("inputs attached:", [p.name for p in KAGGLE_INPUT.iterdir()] if KAGGLE_INPUT.exists() else "n/a")
assert COMP_DIR is not None, "competition data not attached (Add Input -> Competitions -> " + COMP_SLUG + ")"
if MODEL_DIR is None or PKG_DIR is None:
    print("\n" + "!" * 78)
    print("!! Trackastra bundle is NOT attached. Add Input -> Your Work -> 'biohub-prep-bundle' notebook output")
    print("!! (it holds the offline wheels for zarr/trackastra and the pretrained Trackastra weights).")
    print("!" * 78 + "\n")

TEST_ZARRS = sorted(p for p in (COMP_DIR / "test").iterdir() if p.name.endswith(".zarr"))
n_train = sum(1 for p in (COMP_DIR / "train").iterdir() if p.name.endswith(".zarr")) if (COMP_DIR / "train").is_dir() else -1
print(f"{len(TEST_ZARRS)} test samples (hidden set is ~ train size = {n_train}):", [p.name for p in TEST_ZARRS[:4]], "...")
import torch
if not torch.cuda.is_available():
    print("!! No GPU: set Settings -> Accelerator -> GPU (T4/P100).")


In [ ]:
# ------------------------------------------------------------------ 2. offline install (no internet)
NEEDED = {  # import name -> wheel project name
    "trackastra": "trackastra", "geff": "geff", "geff_spec": "geff_spec", "zarr": "zarr",
    "numcodecs": "numcodecs", "donfig": "donfig", "google_crc32c": "google_crc32c",
    "edt": "edt", "lz4": "lz4", "imagecodecs": "imagecodecs", "chardet": "chardet",
    "platformdirs": "platformdirs", "psutil": "psutil", "dask": "dask", "numba": "numba",
    "skimage": "scikit_image", "yaml": "pyyaml", "joblib": "joblib", "tqdm": "tqdm",
    "pandas": "pandas", "scipy": "scipy", "torch": "torch", "torchvision": "torchvision",
    "typing_extensions": "typing_extensions", "pydantic": "pydantic", "requests": "requests",
    "motile": "motile", "ilpy": "ilpy", "pyscipopt": "pyscipopt", "structsvm": "structsvm",  # optional: enables mode="ilp"
    # official-baseline (tracking_cellmot) stack, present only in prep-bundle v2
    "tracksdata": "tracksdata", "polars": "polars", "rustworkx": "rustworkx", "bidict": "bidict", "psygnal": "psygnal",
    "blosc2": "blosc2", "rstar_python": "rstar_python", "sqlalchemy": "sqlalchemy", "pyarrow": "pyarrow", "rich": "rich",
    "ndindex": "ndindex", "msgpack": "msgpack", "numexpr": "numexpr",
}

def _importable(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

def _pip(args):
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-warn-script-location", "-q"] + args
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])
    return r.returncode == 0

def _installed(mod):
    """present on disk? (importlib.util.find_spec does NOT import the module -> nothing stale gets loaded)"""
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

TRACKASTRA_OK = False
if PKG_DIR is not None:
    import importlib.util, importlib.metadata as im
    from packaging.version import Version
    # step A0: Kaggle ships OLD versions of some libs that tracksdata needs newer (polars < 1.36 lacks pl.Float16).
    # Upgrade from the bundle BEFORE anything imports them (re-importing a native lib after upgrade can crash the kernel).
    UPGRADE = ["polars", "polars_runtime_32", "rustworkx", "psygnal", "bidict", "blosc2", "rstar_python", "numcodecs"]
    ups = []
    for proj in UPGRADE:
        w = sorted(glob.glob(str(PKG_DIR / f"{proj}-*.whl")))
        if not w:
            continue
        ver = Path(w[-1]).name.split("-")[1]
        try:
            inst = im.version(proj.replace("_", "-"))
        except Exception:
            inst = None
        if inst is None or Version(inst) < Version(ver):
            ups.append(w[-1]); print(f"    upgrade {proj}: {inst} -> {ver}")
    if ups:
        _pip(["--no-deps", "--upgrade"] + ups); importlib.invalidate_caches()
    # step A: only the missing projects (checked without importing), no dependency resolution (keeps Kaggle torch/numpy intact)
    missing = [proj for mod, proj in NEEDED.items() if not _installed(mod)]
    print("missing before install:", missing)
    wheels = []
    for proj in missing:
        w = sorted(glob.glob(str(PKG_DIR / f"{proj}-*.whl")))
        wheels += w[:1] if w else []
    if wheels:
        _pip(["--no-deps"] + wheels); importlib.invalidate_caches()
    still = [proj for mod, proj in NEEDED.items() if not _importable(mod)]      # now really import to validate
    print("missing after step A:", still)
    if still:
        # step B: let pip resolve, but pin already-installed heavy libs so they are not replaced
        cons = OUT_DIR / "constraints.txt"
        pins = []
        for proj in ["torch", "torchvision", "numpy", "scipy", "pandas", "scikit-image", "numba", "llvmlite"]:
            try: pins.append(f"{proj}=={im.version(proj)}")
            except Exception: pass
        cons.write_text("\n".join(pins))
        extra = ["tracksdata"] if glob.glob(str(PKG_DIR / "tracksdata-*.whl")) else []
        _pip(["--find-links", str(PKG_DIR), "-c", str(cons), "trackastra", "zarr", "geff"] + extra)
        importlib.invalidate_caches()
    TRACKASTRA_OK = _importable("trackastra") and _importable("zarr")
else:
    TRACKASTRA_OK = _importable("trackastra") and _importable("zarr")

import numpy as np, torch
print("python", sys.version.split()[0], "| numpy", np.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(), "| trackastra importable:", TRACKASTRA_OK)
assert _importable("zarr"), ("zarr is not installed and no wheel bundle found -> attach the prep-bundle notebook output "
    "(it ships zarr/numcodecs/trackastra wheels for Python 3.12)")


In [ ]:
# ------------------------------------------------------------------ 3. pipeline code (identical to the inference notebook)
"""
Biohub Cell Tracking - inference pipeline (shared by notebook cells).

Steps per test sample:
  1. load zarr v3 volume (T,Z,Y,X) uint16
  2. per-frame 3D blob detection (DoG + local maxima, anisotropic voxel scale)
  3. spherical pseudo-masks around detections
  4. Trackastra (pretrained CTC 3D transformer) greedy linking with divisions
     -> fallback: scaled nearest-neighbour linear assignment
  5. rows for submission.csv
"""
from __future__ import annotations

import gc
import json
import time
import warnings
from pathlib import Path

import numpy as np

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------- config
SCALE_ZYX = np.array([1.625, 0.40625, 0.40625], dtype=np.float32)  # um / voxel

CFG = dict(
    cell_radius_um=3.5,      # approx nucleus radius, drives DoG sigma & NMS footprint
    detect_percentile=99.8,  # per-frame intensity normalisation
    thresh_k=1.0,            # threshold = mean + k*std of DoG response (foreground only)
    min_rel_peak=0.08,       # peak must exceed this fraction of max DoG response
    max_nodes_per_frame=6000,
    merge_min_dist_um=None,  # if set: within a frame, suppress weaker detections closer than this (um) to a stronger one
    # per-clip adaptive radius: r = clip(a * N^(-1/3), rmin, rmax), N = median detections/frame at probe radius.
    # Rationale: early zebrafish embryo volume is ~constant, so more cells <=> smaller cells. None = fixed radius.
    adaptive_radius=None,    # e.g. dict(a=26.5, rmin=3.0, rmax=6.0, probe_r=4.0, n_probe=5)
    mask_radius_vox=(1, 3, 3),  # (z,y,x) radius of pseudo-mask spheres (used when mask_scale is None)
    mask_scale=None,         # if set: pseudo-mask radius = mask_scale * cell_radius_um (in um), converted per axis
    link_max_dist_um=12.0,   # fallback linker gate
    trackastra_mode="greedy",
    trackastra_max_distance=64,  # voxel (isotropic-ish) neighbourhood for candidate graph
    # division pruning (greedy over-predicts divisions massively on this data):
    div_min_weight=0.5,      # 2nd child edge kept only if its trackastra weight >= this (None = keep all)
    div_max_dist_um=8.0,     # ... and both children within this distance of the parent
    div_max_frac=0.006,      # ... and at most this fraction of nodes may be division parents (keep strongest)
)


# ---------------------------------------------------------------- io
def open_zarr_array(zarr_dir: Path):
    """Return array-like (T,Z,Y,X). Works for zarr v3 (array at '0')."""
    import zarr

    zarr_dir = Path(zarr_dir)
    try:
        g = zarr.open_group(str(zarr_dir), mode="r")
        arr = g["0"]
    except Exception:
        arr = zarr.open_array(str(zarr_dir / "0"), mode="r")
    return arr


def list_test_samples(comp_dir: Path):
    test_dir = Path(comp_dir) / "test"
    zarrs = sorted(p for p in test_dir.iterdir() if p.suffix == ".zarr" and p.is_dir())
    return zarrs


# ---------------------------------------------------------------- detection
def _torch_device():
    import torch

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _gauss_kernel1d(sigma: float, device):
    import torch

    if sigma <= 0.05:
        return torch.ones(1, device=device)
    r = max(1, int(np.ceil(3 * sigma)))
    x = torch.arange(-r, r + 1, device=device, dtype=torch.float32)
    k = torch.exp(-0.5 * (x / sigma) ** 2)
    return k / k.sum()


def _gauss3d(vol, sigmas_zyx):
    """Separable Gaussian blur, vol: torch (Z,Y,X) float32."""
    import torch
    import torch.nn.functional as F

    x = vol[None, None]
    for axis, s in enumerate(sigmas_zyx):
        k = _gauss_kernel1d(float(s), vol.device)
        if k.numel() == 1:
            continue
        shape = [1, 1, 1, 1, 1]
        shape[2 + axis] = k.numel()
        pad = [0, 0, 0, 0, 0, 0]
        # F.pad order: (x_l, x_r, y_l, y_r, z_l, z_r)
        pad_idx = (2 - axis) * 2
        pad[pad_idx] = pad[pad_idx + 1] = k.numel() // 2
        x = F.pad(x, pad, mode="replicate")
        x = F.conv3d(x, k.view(*shape))
    return x[0, 0]


def _detect_frame_cpu(frame_u16: np.ndarray, cfg=CFG):
    """scipy.ndimage version (much faster than torch conv3d on CPU)."""
    from scipy import ndimage as ndi

    v = frame_u16.astype(np.float32)
    sub = v.ravel()[:: max(1, v.size // 200000)]
    lo, hi = np.percentile(sub, 1.0), np.percentile(sub, cfg["detect_percentile"])
    v = np.clip((v - lo) / (hi - lo + 1e-6), 0, 1)

    r_vox = cfg["cell_radius_um"] / SCALE_ZYX
    s1 = r_vox / np.sqrt(3.0) * 0.8
    s2 = s1 * 1.6
    dog = ndi.gaussian_filter(v, s1, mode="nearest") - ndi.gaussian_filter(v, s2, mode="nearest")

    fp = tuple(int(max(1, np.floor(r * 0.8))) * 2 + 1 for r in r_vox)
    mp = ndi.maximum_filter(dog, size=fp, mode="nearest")
    is_peak = dog >= mp

    pos = dog[dog > 0]
    if pos.size < 10:
        return np.zeros((0, 3), np.int32), np.zeros((0,), np.float32)
    thr = float(pos.mean() + cfg["thresh_k"] * pos.std())
    thr = max(thr, cfg["min_rel_peak"] * float(dog.max()))
    cand = is_peak & (dog > thr)
    idx = np.argwhere(cand)
    scores = dog[cand]
    if idx.shape[0] > cfg["max_nodes_per_frame"]:
        top = np.argsort(-scores)[: cfg["max_nodes_per_frame"]]
        idx, scores = idx[top], scores[top]
    return idx.astype(np.int32), scores.astype(np.float32)


def detect_frame(frame_u16: np.ndarray, cfg=CFG, device=None):
    """Detect cell centroids in one (Z,Y,X) frame. Returns int array (N,3) zyx + scores (N,)."""
    import torch
    import torch.nn.functional as F

    device = device or _torch_device()
    if device.type == "cpu":
        return _detect_frame_cpu(frame_u16, cfg)
    v = torch.from_numpy(frame_u16.astype(np.float32)).to(device)

    lo = torch.quantile(v.flatten()[:: max(1, v.numel() // 200000)], 0.01)
    hi = torch.quantile(v.flatten()[:: max(1, v.numel() // 200000)], cfg["detect_percentile"] / 100.0)
    v = ((v - lo) / (hi - lo + 1e-6)).clamp_(0, 1)

    r_vox = cfg["cell_radius_um"] / SCALE_ZYX  # radius in voxels per axis (z,y,x)
    s1 = r_vox / np.sqrt(3.0) * 0.8
    s2 = s1 * 1.6
    dog = _gauss3d(v, s1) - _gauss3d(v, s2)

    # local maxima via max-pool with anisotropic footprint (~ cell radius)
    fp = [int(max(1, np.floor(r * 0.8))) * 2 + 1 for r in r_vox]
    mp = F.max_pool3d(dog[None, None], kernel_size=fp, stride=1, padding=[f // 2 for f in fp])[0, 0]
    is_peak = dog >= mp

    pos = dog[dog > 0]
    if pos.numel() < 10:
        return np.zeros((0, 3), np.int32), np.zeros((0,), np.float32)
    thr = float(pos.mean() + cfg["thresh_k"] * pos.std())
    thr = max(thr, cfg["min_rel_peak"] * float(dog.max()))
    cand = is_peak & (dog > thr)

    idx = torch.nonzero(cand, as_tuple=False)
    scores = dog[cand]
    if idx.shape[0] > cfg["max_nodes_per_frame"]:
        top = torch.topk(scores, cfg["max_nodes_per_frame"]).indices
        idx, scores = idx[top], scores[top]
    return idx.cpu().numpy().astype(np.int32), scores.cpu().numpy().astype(np.float32)


def merge_close_detections(zyx, scores, min_dist_um):
    """Greedy NMS on physical distance: keep strongest, drop any weaker detection within min_dist_um."""
    if min_dist_um is None or len(zyx) < 2:
        return zyx
    from scipy.spatial import cKDTree

    P_ = zyx.astype(np.float32) * SCALE_ZYX
    order = np.argsort(-scores)
    tree = cKDTree(P_)
    alive = np.ones(len(zyx), bool)
    for i in order:
        if not alive[i]:
            continue
        for j in tree.query_ball_point(P_[i], min_dist_um):
            if j != i and alive[j] and scores[j] <= scores[i]:
                alive[j] = False
    return zyx[alive]


def detect_volume(arr, cfg=CFG, device=None, log=print):
    """arr: (T,Z,Y,X) array-like. Returns list of (N_t,3) int zyx arrays and imgs float16 (T,Z,Y,X)."""
    T = arr.shape[0]
    dets, imgs = [], []
    t0 = time.time()
    for t in range(T):
        frame = np.asarray(arr[t])
        zyx, sc = detect_frame(frame, cfg, device)
        zyx = merge_close_detections(zyx, sc, cfg.get("merge_min_dist_um"))
        dets.append(zyx)
        imgs.append(frame)
        if t % 25 == 0 or t == T - 1:
            log(f"    frame {t:3d}/{T}: {len(zyx):5d} detections  ({time.time()-t0:.1f}s)")
    return dets, np.stack(imgs)


# ---------------------------------------------------------------- masks
def _sphere_offsets(radius_zyx):
    rz, ry, rx = radius_zyx
    zz, yy, xx = np.mgrid[-rz : rz + 1, -ry : ry + 1, -rx : rx + 1]
    inside = (zz / (rz + 0.5)) ** 2 + (yy / (ry + 0.5)) ** 2 + (xx / (rx + 0.5)) ** 2 <= 1.0
    return np.stack([zz[inside], yy[inside], xx[inside]], axis=1)


def make_masks(dets, shape_zyx, radius_zyx=CFG["mask_radius_vox"]):
    """Label volume (T,Z,Y,X) int32; label = 1..N_t per frame (matches detection order)."""
    T = len(dets)
    Z, Y, X = shape_zyx
    masks = np.zeros((T, Z, Y, X), dtype=np.int32)
    off = _sphere_offsets(radius_zyx)
    for t, zyx in enumerate(dets):
        if len(zyx) == 0:
            continue
        pts = (zyx[:, None, :] + off[None, :, :]).reshape(-1, 3)
        labels = np.repeat(np.arange(1, len(zyx) + 1, dtype=np.int32), len(off))
        ok = (
            (pts[:, 0] >= 0) & (pts[:, 0] < Z)
            & (pts[:, 1] >= 0) & (pts[:, 1] < Y)
            & (pts[:, 2] >= 0) & (pts[:, 2] < X)
        )
        pts, labels = pts[ok], labels[ok]
        # later labels overwrite earlier -> fine (centre voxel always set last below)
        masks[t, pts[:, 0], pts[:, 1], pts[:, 2]] = labels
        masks[t, zyx[:, 0], zyx[:, 1], zyx[:, 2]] = np.arange(1, len(zyx) + 1, dtype=np.int32)
    return masks


# ---------------------------------------------------------------- linking
def link_trackastra(model, imgs, masks, dets, cfg=CFG, log=print):
    """Returns (nodes: list of (t, z, y, x, node_id), edges: list of (src_id, tgt_id))."""
    t0 = time.time()
    graph, _ = model.track(
        imgs,
        masks,
        mode=cfg["trackastra_mode"],
        max_distance=cfg["trackastra_max_distance"],
    )
    log(f"    trackastra: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges ({time.time()-t0:.1f}s)")

    # map (time,label) -> our detection coordinate
    key2id = {}
    nodes = []
    nid = 1
    for t, zyx in enumerate(dets):
        for lab in range(1, len(zyx) + 1):
            key2id[(t, lab)] = nid
            z, y, x = zyx[lab - 1]
            nodes.append((t, int(z), int(y), int(x), nid))
            nid += 1

    edges, weights = [], []
    for u, v, d in graph.edges(data=True):
        nu, nv = graph.nodes[u], graph.nodes[v]
        ku = (int(nu["time"]), int(nu["label"]))
        kv = (int(nv["time"]), int(nv["label"]))
        if ku in key2id and kv in key2id:
            edges.append((key2id[ku], key2id[kv]))
            weights.append(float(d.get("weight", 1.0)))
    edges = prune_divisions(nodes, edges, weights, cfg, log=log)
    return nodes, edges


def prune_divisions(nodes, edges, weights, cfg=CFG, log=print):
    """Keep at most one outgoing edge per parent unless the 2nd edge is confident, close, and
    the global division budget (div_max_frac * n_nodes) is not exceeded."""
    if cfg.get("div_min_weight") is None:
        return edges
    pos = {nid: np.array([z, y, x], np.float32) * SCALE_ZYX for t, z, y, x, nid in nodes}
    by_parent = {}
    for (s, t_), w in zip(edges, weights):
        by_parent.setdefault(s, []).append((w, t_))
    n_div_before = sum(1 for c in by_parent.values() if len(c) >= 2)
    kept, candidates = [], []
    for s, ch in by_parent.items():
        ch.sort(reverse=True)
        kept.append((s, ch[0][1]))                      # strongest child always kept
        for w, t_ in ch[1:2]:                            # at most one extra child
            d1 = np.linalg.norm(pos[ch[0][1]] - pos[s])
            d2 = np.linalg.norm(pos[t_] - pos[s])
            if w >= cfg["div_min_weight"] and max(d1, d2) <= cfg["div_max_dist_um"]:
                candidates.append((w, s, t_))
    budget = int(cfg["div_max_frac"] * len(nodes))
    candidates.sort(reverse=True)
    kept += [(s, t_) for _, s, t_ in candidates[:budget]]
    log(f"    divisions: {n_div_before} -> {min(len(candidates), budget)} after pruning "
        f"(min_w={cfg['div_min_weight']}, max_d={cfg['div_max_dist_um']}um, budget={budget})")
    return kept


def link_nearest(dets, cfg=CFG, log=print):
    """Fallback: frame-to-frame linear assignment on physical distance, + division by
    attaching unmatched t+1 nodes to the nearest matched parent within the gate."""
    from scipy.optimize import linear_sum_assignment
    from scipy.spatial import cKDTree

    gate = cfg["link_max_dist_um"]
    nodes, edges = [], []
    ids = []
    nid = 1
    for t, zyx in enumerate(dets):
        cur = np.arange(nid, nid + len(zyx))
        ids.append(cur)
        for (z, y, x), i in zip(zyx, cur):
            nodes.append((t, int(z), int(y), int(x), int(i)))
        nid += len(zyx)

    for t in range(len(dets) - 1):
        a, b = dets[t].astype(np.float32) * SCALE_ZYX, dets[t + 1].astype(np.float32) * SCALE_ZYX
        if len(a) == 0 or len(b) == 0:
            continue
        tree = cKDTree(a)
        d, j = tree.query(b, k=min(5, len(a)), distance_upper_bound=gate)
        d = np.atleast_2d(d.T).T if d.ndim == 1 else d
        j = np.atleast_2d(j.T).T if j.ndim == 1 else j
        big = 1e6
        cost = np.full((len(a), len(b)), big, np.float32)
        for bi in range(len(b)):
            for dd, aj in zip(d[bi], j[bi]):
                if np.isfinite(dd) and aj < len(a):
                    cost[aj, bi] = dd
        r, c = linear_sum_assignment(cost)
        matched_b = set()
        for ai, bi in zip(r, c):
            if cost[ai, bi] < big:
                edges.append((int(ids[t][ai]), int(ids[t + 1][bi])))
                matched_b.add(bi)
        # divisions: unmatched children -> nearest parent within gate
        parent_children = {}
        for ai, bi in zip(r, c):
            if cost[ai, bi] < big:
                parent_children[ai] = 1
        for bi in range(len(b)):
            if bi in matched_b:
                continue
            dd, aj = d[bi][0], j[bi][0]
            if np.isfinite(dd) and aj < len(a) and parent_children.get(aj, 0) < 2:
                edges.append((int(ids[t][aj]), int(ids[t + 1][bi])))
                parent_children[aj] = parent_children.get(aj, 0) + 1
    log(f"    nearest-neighbour linker: {len(nodes)} nodes, {len(edges)} edges")
    return nodes, edges


# ---------------------------------------------------------------- validation on train (.geff)
def read_geff(geff_dir: Path):
    """Return dict(ids, t, z, y, x, edges) from a .geff (zarr v2 or v3)."""
    import zarr

    g = zarr.open_group(str(geff_dir), mode="r")
    out = dict(ids=np.asarray(g["nodes/ids"][:]))
    for k in ("t", "z", "y", "x"):
        out[k] = np.asarray(g[f"nodes/props/{k}/values"][:])
    try:
        out["edges"] = np.asarray(g["edges/ids"][:]).reshape(-1, 2)
    except Exception:
        out["edges"] = np.zeros((0, 2), np.int64)
    def _find(obj, key):
        if isinstance(obj, dict):
            if key in obj:
                return obj[key]
            for v in obj.values():
                r = _find(v, key)
                if r is not None:
                    return r
        elif isinstance(obj, list):
            for v in obj:
                r = _find(v, key)
                if r is not None:
                    return r
        return None
    try:
        out["estimated_number_of_nodes"] = _find(dict(g.attrs), "estimated_number_of_nodes")
    except Exception:
        out["estimated_number_of_nodes"] = None
    return out


def evaluate_against_geff(nodes, edges, gt, match_um=7.0):
    """Sparse-GT diagnostics: node recall, edge recall/precision-on-matched, divisions.
    (Official metric is more involved; this is a sanity check only.)"""
    from scipy.optimize import linear_sum_assignment

    pred_by_t = {}
    for t, z, y, x, nid in nodes:
        pred_by_t.setdefault(t, []).append((nid, z, y, x))
    gt_by_t = {}
    for i, t, z, y, x in zip(gt["ids"], gt["t"], gt["z"], gt["y"], gt["x"]):
        gt_by_t.setdefault(int(t), []).append((int(i), z, y, x))

    pred2gt, n_gt, n_hit = {}, 0, 0
    for t, gl in gt_by_t.items():
        pl = pred_by_t.get(t, [])
        n_gt += len(gl)
        if not pl:
            continue
        P_ = np.array([p[1:] for p in pl], np.float32) * SCALE_ZYX
        G_ = np.array([g[1:] for g in gl], np.float32) * SCALE_ZYX
        D = np.linalg.norm(P_[:, None, :] - G_[None, :, :], axis=2)
        r, c = linear_sum_assignment(D)
        for i, j in zip(r, c):
            if D[i, j] <= match_um:
                pred2gt[pl[i][0]] = gl[j][0]
                n_hit += 1

    gt_edges = set(map(tuple, gt["edges"].tolist()))
    gt2pred = {v: k for k, v in pred2gt.items()}
    pred_edges = set(edges)
    # edge recall: GT edges whose both endpoints were matched and the predicted edge exists
    e_hit = e_eval = 0
    for s, t_ in gt_edges:
        if s in gt2pred and t_ in gt2pred:
            e_eval += 1
            if (gt2pred[s], gt2pred[t_]) in pred_edges:
                e_hit += 1
    # divisions
    def _divs(E):
        cnt = {}
        for s, _ in E:
            cnt[s] = cnt.get(s, 0) + 1
        return {s for s, c in cnt.items() if c >= 2}

    gt_div = _divs(gt_edges)
    pred_div = _divs(pred_edges)
    div_hit = sum(1 for s in gt_div if gt2pred.get(s) in pred_div)
    return dict(
        gt_nodes=n_gt,
        node_recall=n_hit / max(1, n_gt),
        gt_edges_evaluable=e_eval,
        edge_recall_on_matched=e_hit / max(1, e_eval),
        gt_edges_total=len(gt_edges),
        edge_recall_total=e_hit / max(1, len(gt_edges)),
        gt_divisions=len(gt_div),
        pred_divisions=len(pred_div),
        division_recall=div_hit / max(1, len(gt_div)),
        pred_nodes=len(nodes),
        est_total_cells=gt.get("estimated_number_of_nodes"),
        pred_over_est=(len(nodes) / gt["estimated_number_of_nodes"]) if gt.get("estimated_number_of_nodes") else None,
    )


# ---------------------------------------------------------------- official-style metric (local CV)
def score_official(nodes, edges, gt, match_um=7.0, a=0.1, w=0.1):
    """Re-implementation of the competition metric (metrics.md):
    - nodes matched per timepoint by optimal bipartite assignment, <= 7 um
    - edge TP: both endpoints matched to GT nodes joined by a GT edge
      edge FP: predicted edge whose matched source has a *different* GT target, or whose matched target has a different GT source
      edge FN: GT edges not recovered
    - adjusted J = max(0, J * (1 - a*(T_pred - T_true)/T_true))  with T_true = estimated_number_of_nodes
    - division TP/FP/FN (simplified: parent matched + both daughters matched & linked)
    Returns dict with counts and score."""
    from scipy.optimize import linear_sum_assignment

    pred_by_t, gt_by_t = {}, {}
    for t, z, y, x, nid in nodes:
        pred_by_t.setdefault(int(t), []).append((nid, z, y, x))
    for i, t, z, y, x in zip(gt["ids"], gt["t"], gt["z"], gt["y"], gt["x"]):
        gt_by_t.setdefault(int(t), []).append((int(i), z, y, x))
    p2g = {}
    for t, gl in gt_by_t.items():
        pl = pred_by_t.get(t, [])
        if not pl:
            continue
        P_ = np.array([p[1:] for p in pl], np.float32) * SCALE_ZYX
        G_ = np.array([g[1:] for g in gl], np.float32) * SCALE_ZYX
        D = np.linalg.norm(P_[:, None, :] - G_[None, :, :], axis=2)
        r, c = linear_sum_assignment(D)
        for i, j in zip(r, c):
            if D[i, j] <= match_um:
                p2g[pl[i][0]] = gl[j][0]
    g2p = {v: k for k, v in p2g.items()}
    n_gt = len(gt["ids"]); n_matched = len(p2g)

    gt_edges = set(map(tuple, gt["edges"].tolist()))
    gt_out, gt_in = {}, {}
    for s_, t_ in gt_edges:
        gt_out.setdefault(s_, set()).add(t_)
        gt_in.setdefault(t_, set()).add(s_)

    tp = fp = 0
    pred_edges = set(edges)
    for s_, t_ in pred_edges:
        gs, gtn = p2g.get(s_), p2g.get(t_)
        if gs is not None and gtn is not None and gtn in gt_out.get(gs, ()):
            tp += 1
            continue
        # FP: matched source whose GT target is someone else, or matched target whose GT source is someone else
        if (gs is not None and gs in gt_out and gtn not in gt_out[gs]) or \
           (gtn is not None and gtn in gt_in and gs not in gt_in[gtn]):
            fp += 1
    fn = len(gt_edges) - tp
    J = tp / max(1, tp + fp + fn)
    est = gt.get("estimated_number_of_nodes")
    adj = J
    if est:
        adj = max(0.0, J * (1 - a * (len(nodes) - est) / est))

    # divisions
    gt_div = {s_ for s_, ch in gt_out.items() if len(ch) >= 2}
    pred_out = {}
    for s_, t_ in pred_edges:
        pred_out.setdefault(s_, set()).add(t_)
    pred_div = {s_ for s_, ch in pred_out.items() if len(ch) >= 2}
    dtp = 0
    for gp in gt_div:
        pp = g2p.get(gp)
        if pp is None:
            continue
        daughters = gt_out[gp]
        if all(g2p.get(d) in pred_out.get(pp, ()) for d in daughters):
            dtp += 1
    # FP divisions: predicted division whose parent is a matched GT node that does not divide in GT
    dfp = sum(1 for pp in pred_div if pp in p2g and p2g[pp] not in gt_div)
    dfn = len(gt_div) - dtp
    dJ = dtp / max(1, dtp + dfp + dfn)
    return dict(edge_tp=tp, edge_fp=fp, edge_fn=fn, edge_J=round(J, 4), adj_edge_J=round(adj, 4),
                div_tp=dtp, div_fp=dfp, div_fn=dfn, div_J=round(dJ, 4),
                score=round(adj + w * dJ, 4), node_recall=round(n_matched / max(1, n_gt), 4),
                n_pred=len(nodes), est=est, gt_div=len(gt_div))


def micro_average(rows, w=0.1):
    """Combine per-clip dicts from score_official the way the competition does (micro over videos)."""
    tp = sum(r["edge_tp"] for r in rows); fp = sum(r["edge_fp"] for r in rows); fn = sum(r["edge_fn"] for r in rows)
    dtp = sum(r["div_tp"] for r in rows); dfp = sum(r["div_fp"] for r in rows); dfn = sum(r["div_fn"] for r in rows)
    # per-sample adjusted J weighted by (TP+FP+FN), as described on the competition page
    wts = [r["edge_tp"] + r["edge_fp"] + r["edge_fn"] for r in rows]
    adj = sum(r["adj_edge_J"] * wt for r, wt in zip(rows, wts)) / max(1, sum(wts))
    dJ = dtp / max(1, dtp + dfp + dfn)
    return dict(edge_J=round(tp / max(1, tp + fp + fn), 4), adj_edge_J=round(adj, 4), div_J=round(dJ, 4),
                score=round(adj + w * dJ, 4), edge_tp=tp, edge_fp=fp, edge_fn=fn)


def run_cv(train_zarrs, model, configs, log=print):
    """Evaluate several CFG variants on the same train clips with the official-style metric.
    configs: dict name -> dict of CFG overrides. Returns list of rows (one per config x clip) and summary."""
    results, summary = [], []
    gts = {zp: read_geff(zp.parent / (zp.name[:-5] + ".geff")) for zp in train_zarrs}
    for name, over in configs.items():
        cfg = dict(CFG); cfg.update(over)
        rows = []
        for zp in train_zarrs:
            r_, info = process_sample(zp, model, cfg, log=lambda *a: None)
            nodes = [(r[3], r[4], r[5], r[6], r[2]) for r in r_ if r[1] == "node"]
            edges = [(r[7], r[8]) for r in r_ if r[1] == "edge"]
            sc = score_official(nodes, edges, gts[zp])
            sc.update(config=name, sample=info["name"])
            rows.append(sc)
            log(f"    [{name}] {info['name']}: score={sc['score']:.3f} adjJ={sc['adj_edge_J']:.3f} "
                f"(tp={sc['edge_tp']} fp={sc['edge_fp']} fn={sc['edge_fn']}) node_recall={sc['node_recall']:.3f} "
                f"divJ={sc['div_J']:.2f} (gt_div={sc['gt_div']}) n_pred/est={sc['n_pred']}/{sc['est']}")
        ma = micro_average(rows); ma["config"] = name
        summary.append(ma)
        log(f"  ==> [{name}] micro score={ma['score']:.4f} adjJ={ma['adj_edge_J']:.4f} divJ={ma['div_J']:.3f}")
        results += rows
    return results, summary


# ---------------------------------------------------------------- detection calibration on train
def calibrate_thresh_k(train_zarrs, cfg=CFG, ks=(0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0), log=print):
    """Sweep thresh_k on a few train clips; pick the k whose total detections best match the
    estimated_number_of_nodes from the .geff metadata (ratio closest to 1). Also reports GT node recall.
    Returns (best_k, table) or (None, table) when no estimate is available."""
    import torch
    from scipy.optimize import linear_sum_assignment

    rows = []
    frames_cache = []
    for zp in train_zarrs:
        gt = read_geff(zp.parent / (zp.name[:-5] + ".geff"))
        est = gt.get("estimated_number_of_nodes")
        arr = open_zarr_array(zp)
        T = arr.shape[0]
        # evaluate recall on a subset of frames for speed; count on all frames
        frames_cache.append((zp.name[:-5], arr, gt, est))
    for k in ks:
        c = dict(cfg); c["thresh_k"] = k
        for name, arr, gt, est in frames_cache:
            n_det, n_hit, n_gt = 0, 0, 0
            gt_by_t = {}
            for i, t, z, y, x in zip(gt["ids"], gt["t"], gt["z"], gt["y"], gt["x"]):
                gt_by_t.setdefault(int(t), []).append((z, y, x))
            for t in range(arr.shape[0]):
                zyx, sc_ = detect_frame(np.asarray(arr[t]), c)
                zyx = merge_close_detections(zyx, sc_, c.get("merge_min_dist_um"))
                n_det += len(zyx)
                gl = gt_by_t.get(t)
                if gl and len(zyx):
                    P_ = zyx.astype(np.float32) * SCALE_ZYX
                    G_ = np.array(gl, np.float32) * SCALE_ZYX
                    D = np.linalg.norm(P_[:, None, :] - G_[None, :, :], axis=2)
                    r, cidx = linear_sum_assignment(D)
                    n_hit += int((D[r, cidx] <= 7.0).sum())
                    n_gt += len(gl)
                elif gl:
                    n_gt += len(gl)
            rows.append(dict(sample=name, thresh_k=k, n_det=n_det, est=est,
                             ratio=(n_det / est) if est else None, node_recall=n_hit / max(1, n_gt)))
            log(f"    k={k:<4} {name}: det={n_det:6d} est={est} ratio={rows[-1]['ratio'] and round(rows[-1]['ratio'],3)} recall={rows[-1]['node_recall']:.3f}")
    with_est = [r for r in rows if r["ratio"] is not None]
    if not with_est:
        log("    no estimated_number_of_nodes in geff metadata -> keeping default thresh_k")
        return None, rows
    best, best_err = None, 1e9
    for k in ks:
        rs = [r for r in with_est if r["thresh_k"] == k]
        err = float(np.median([abs(np.log(r["ratio"])) for r in rs]))   # log-ratio: over/under symmetric
        if err < best_err:
            best, best_err = k, err
    log(f"    -> best thresh_k = {best} (median |log ratio| = {best_err:.3f})")
    return best, rows


# ---------------------------------------------------------------- submission
def rows_for_sample(dataset: str, nodes, edges):
    rows = []
    for t, z, y, x, nid in nodes:
        rows.append((dataset, "node", nid, t, z, y, x, -1, -1))
    for s, tgt in edges:
        rows.append((dataset, "edge", -1, -1, -1, -1, -1, s, tgt))
    return rows


def write_submission(all_rows, out_path: Path):
    import pandas as pd

    df = pd.DataFrame(
        all_rows, columns=["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    )
    df.insert(0, "id", np.arange(len(df)))
    for c in ["id", "node_id", "t", "z", "y", "x", "source_id", "target_id"]:
        df[c] = df[c].astype(np.int64)
    df.to_csv(out_path, index=False)
    return df


def choose_radius(arr, cfg=CFG, log=print):
    """Probe a few frames at probe_r, take the median detection count, map to a radius."""
    ar = cfg.get("adaptive_radius")
    if not ar:
        return cfg["cell_radius_um"]
    probe = dict(cfg); probe["cell_radius_um"] = ar.get("probe_r", 4.0); probe["adaptive_radius"] = None
    T = arr.shape[0]
    idx = np.linspace(0, T - 1, ar.get("n_probe", 5)).round().astype(int)
    counts = []
    for t in idx:
        zyx, sc = detect_frame(np.asarray(arr[t]), probe)
        zyx = merge_close_detections(zyx, sc, probe.get("merge_min_dist_um"))
        counts.append(len(zyx))
    n = max(1.0, float(np.median(counts)))
    r = float(np.clip(ar.get("a", 26.5) * n ** (-1.0 / 3.0), ar.get("rmin", 3.0), ar.get("rmax", 6.0)))
    r = round(r * 4) / 4  # quarter-um steps
    log(f"    adaptive radius: median {n:.0f} cells/frame at probe r={probe['cell_radius_um']} -> r={r} um")
    return r


def process_sample(zarr_path: Path, model, cfg=CFG, log=print):
    name = zarr_path.name.replace(".zarr", "")
    log(f"[{name}]")
    arr = open_zarr_array(zarr_path)
    log(f"    shape={arr.shape} dtype={arr.dtype}")
    cfg = dict(cfg)
    if cfg.get("adaptive_radius"):
        cfg["cell_radius_um"] = choose_radius(arr, cfg, log=log)
    if cfg.get("mask_scale"):
        r_um = cfg["mask_scale"] * cfg["cell_radius_um"]
        cfg["mask_radius_vox"] = tuple(int(max(1, round(r_um / s_))) for s_ in SCALE_ZYX)
        log(f"    pseudo-mask radius (z,y,x) vox = {cfg['mask_radius_vox']}  (~{r_um:.1f} um)")
    dets, imgs = detect_volume(arr, cfg, log=log)
    n_det = sum(len(d) for d in dets)
    log(f"    total detections: {n_det}")
    nodes, edges = None, None
    if model is not None and n_det > 0:
        try:
            masks = make_masks(dets, arr.shape[1:], cfg["mask_radius_vox"])
            nodes, edges = link_trackastra(model, imgs, masks, dets, cfg, log=log)
            del masks
        except Exception as e:  # noqa
            log(f"    !! trackastra failed ({type(e).__name__}: {e}); falling back to NN linker")
            nodes = None
    if nodes is None:
        nodes, edges = link_nearest(dets, cfg, log=log)
    del imgs
    gc.collect()
    return rows_for_sample(name, nodes, edges), dict(name=name, n_nodes=len(nodes), n_edges=len(edges))


In [ ]:
# ------------------------------------------------------------------ 3b. fine-tuning code
"""
Biohub Cell Tracking - Trackastra fine-tuning on the competition's sparse GT.

Idea
----
The pretrained Trackastra "ctc" linker is kept as is (architecture, features, window=4, quiet_softmax).
We re-train its weights on OUR detections (same DoG pipeline as inference) using the sparse .geff GT:

  * detections are matched to GT nodes per frame (bipartite, <= 7 um, exactly like the metric)
  * a row i (frame t) is supervised only if detection i is matched to a GT node whose GT successor(s)
    at t+1 are ALSO matched to detections  -> target row = one-hot on the matched successor(s), 0 elsewhere
  * a column j (frame t+1) is supervised only if j is matched to a GT node whose GT predecessor is matched
    -> target column = one-hot on the matched predecessor
  * every other pair is masked out of the loss (we simply do not know the answer for unlabeled cells)

This gives exact supervision from sparse labels: within a supervised row, *all* other next-frame detections
are true negatives, because a GT cell has exactly one successor (two for a division).

Everything below is importable both in the notebook (inlined) and locally (tests on synthetic data).
Requires the objects from pipeline.py in the same namespace (CFG, SCALE_ZYX, open_zarr_array, detect_volume,
make_masks, choose_radius, read_geff, score_official, micro_average, prune_divisions).
"""
from __future__ import annotations

import gc
import json
import math
import time
from collections import OrderedDict
from pathlib import Path

import numpy as np

# ------------------------------------------------------------------ feature cache (one npz per clip)
def extract_clip_cache(zarr_path: Path, geff_path: Path, cfg, out_npz: Path, n_workers: int = 0, log=print):
    """Run detection + Trackastra feature extraction exactly like inference, match to GT, save npz."""
    import torch
    from trackastra.data import get_features
    from trackastra.data.wrfeat import WRFeatures
    from trackastra.utils import normalize
    from scipy.optimize import linear_sum_assignment

    t0 = time.time()
    name = zarr_path.name[:-5]
    arr = open_zarr_array(zarr_path)
    cfg = dict(cfg)
    if cfg.get("adaptive_radius"):
        cfg["cell_radius_um"] = choose_radius(arr, cfg, log=log)
    if cfg.get("mask_scale"):
        r_um = cfg["mask_scale"] * cfg["cell_radius_um"]
        cfg["mask_radius_vox"] = tuple(int(max(1, round(r_um / s_))) for s_ in SCALE_ZYX)
    dets, imgs = detect_volume(arr, cfg, log=lambda *a: None)
    masks = make_masks(dets, arr.shape[1:], cfg["mask_radius_vox"])
    imgs = normalize(imgs)                       # identical to Trackastra._predict(normalize_imgs=True)
    feats = get_features(masks, imgs, features="wrfeat", ndim=3, n_workers=n_workers,
                         progbar_class=lambda x, **k: x)
    del masks, imgs
    F = WRFeatures.concat(feats)
    keys = list(F.features.keys()); dims = [int(v.shape[1]) for v in F.features.values()]

    # align feature rows with detections: regionprops sorts labels ascending, label = det index + 1
    det_zyx = np.zeros((len(F), 3), np.int32)
    for t in range(len(dets)):
        sel = np.where(F.timepoints == t)[0]
        labs = F.labels[sel]
        assert len(sel) == len(dets[t]) and np.array_equal(np.sort(labs), np.arange(1, len(dets[t]) + 1)), \
            f"{name}: label/detection mismatch at t={t}"
        det_zyx[sel] = dets[t][labs - 1]

    # GT matching per frame (same rule as the metric: optimal assignment, <= 7 um)
    gt = read_geff(geff_path)
    gt_id = np.full(len(F), -1, np.int64)
    gt_by_t = {}
    for gi, gt_, gz, gy, gx in zip(gt["ids"], gt["t"], gt["z"], gt["y"], gt["x"]):
        gt_by_t.setdefault(int(gt_), []).append((int(gi), gz, gy, gx))
    n_hit = 0
    for t, gl in gt_by_t.items():
        sel = np.where(F.timepoints == t)[0]
        if len(sel) == 0:
            continue
        P_ = det_zyx[sel].astype(np.float32) * SCALE_ZYX
        G_ = np.array([g[1:] for g in gl], np.float32) * SCALE_ZYX
        D = np.linalg.norm(P_[:, None, :] - G_[None, :, :], axis=2)
        r, c = linear_sum_assignment(D)
        for i, j in zip(r, c):
            if D[i, j] <= 7.0:
                gt_id[sel[i]] = gl[j][0]; n_hit += 1
    np.savez_compressed(
        out_npz, name=name, coords=F.coords.astype(np.float32), labels=F.labels.astype(np.int32),
        timepoints=F.timepoints.astype(np.int32), feats=F.features_stacked.astype(np.float32),
        det_zyx=det_zyx, gt_id=gt_id, gt_ids=np.asarray(gt["ids"], np.int64), gt_t=np.asarray(gt["t"], np.int32),
        gt_edges=np.asarray(gt["edges"], np.int64).reshape(-1, 2), est=np.int64(gt.get("estimated_number_of_nodes") or 0),
        radius=np.float32(cfg["cell_radius_um"]), keys=np.array(keys), dims=np.array(dims, np.int32),
        n_frames=np.int32(arr.shape[0]))
    log(f"    [{name}] {len(F)} detections in {arr.shape[0]} frames, r={cfg['cell_radius_um']} um, "
        f"GT nodes matched {n_hit}/{len(gt['ids'])}, GT edges {len(gt['edges'])} ({time.time()-t0:.0f}s)")
    del F, feats, dets
    gc.collect()
    return out_npz


# ------------------------------------------------------------------ clip cache -> supervision
class ClipCache:
    """Per-clip detections + features + GT-derived supervision targets."""

    def __init__(self, npz_path: Path):
        d = np.load(npz_path, allow_pickle=False)
        self.name = str(d["name"]); self.coords = d["coords"]; self.labels = d["labels"]
        self.timepoints = d["timepoints"]; self.feats = d["feats"]; self.det_zyx = d["det_zyx"]
        self.gt_id = d["gt_id"]; self.gt_edges = d["gt_edges"]; self.est = int(d["est"]) or None
        self.keys = [str(k) for k in d["keys"]]; self.dims = [int(x) for x in d["dims"]]
        self.n_frames = int(d["n_frames"]); self.gt_ids = d["gt_ids"]; self.gt_t = d["gt_t"]
        self.radius = float(d["radius"])
        # frame -> row slice (rows are sorted by time then label)
        order = np.lexsort((self.labels, self.timepoints))
        assert np.array_equal(order, np.arange(len(order))), "cache rows not sorted by (t,label)"
        self.frame_start = np.searchsorted(self.timepoints, np.arange(self.n_frames + 1))
        self._build_targets()

    def rows(self, t):
        return np.arange(self.frame_start[t], self.frame_start[t + 1])

    def _build_targets(self):
        gt2row = {int(g): i for i, g in enumerate(self.gt_id) if g >= 0}
        succ, pred = {}, {}
        for s_, t_ in self.gt_edges.tolist():
            succ.setdefault(s_, []).append(t_); pred.setdefault(t_, []).append(s_)
        n = len(self.gt_id)
        # row target: list of successor rows (all successors matched) or None
        self.row_succ = [None] * n
        self.col_pred = np.full(n, -1, np.int64)
        for i, g in enumerate(self.gt_id):
            g = int(g)
            if g < 0:
                continue
            ch = succ.get(g)
            if ch and all(c in gt2row for c in ch):
                rows_ = [gt2row[c] for c in ch]
                if all(self.timepoints[r] == self.timepoints[i] + 1 for r in rows_):
                    self.row_succ[i] = rows_
            pa = pred.get(g)
            if pa and len(pa) == 1 and pa[0] in gt2row and self.timepoints[gt2row[pa[0]]] == self.timepoints[i] - 1:
                self.col_pred[i] = gt2row[pa[0]]
        self.n_sup_rows = sum(1 for r in self.row_succ if r is not None)
        self.n_sup_cols = int((self.col_pred >= 0).sum())
        # frames where a window starting at t1 has supervision
        sup_t = set()
        for i, r in enumerate(self.row_succ):
            if r is not None:
                sup_t.add(int(self.timepoints[i]))          # pair (t, t+1)
        for j in np.where(self.col_pred >= 0)[0]:
            sup_t.add(int(self.timepoints[j]) - 1)
        self.sup_pairs_t = sorted(sup_t)

    def window_starts(self, W):
        starts = set()
        for t in self.sup_pairs_t:                          # pair (t,t+1) is inside window [t1, t1+W) if t1<=t<t1+W-1
            for t1 in range(max(0, t - W + 2), min(t, self.n_frames - W) + 1):
                starts.add(t1)
        return sorted(starts)

    def wrfeatures(self, rows):
        from trackastra.data.wrfeat import WRFeatures
        fd, o = OrderedDict(), 0
        for k, dd in zip(self.keys, self.dims):
            fd[k] = self.feats[rows, o:o + dd]; o += dd
        return WRFeatures(coords=self.coords[rows].copy(), labels=self.labels[rows], timepoints=self.timepoints[rows], features=fd)

    def frame_wrfeatures(self):
        return [self.wrfeatures(self.rows(t)) for t in range(self.n_frames)]

    def make_window(self, t1, W, augmenter=None, max_tokens=None):
        """Returns dict(coords (n,4) float, features (n,F), timepoints (n,), A (n,n), M (n,n)) as torch tensors."""
        import torch
        rows = np.arange(self.frame_start[t1], self.frame_start[t1 + W])
        if max_tokens and len(rows) > max_tokens:      # drop whole frames from the end (like trackastra)
            while len(rows) > max_tokens and W > 2:
                W -= 1; rows = np.arange(self.frame_start[t1], self.frame_start[t1 + W])
        n = len(rows)
        A = np.zeros((n, n), np.float32); M = np.zeros((n, n), np.float32)
        loc = {int(r): k for k, r in enumerate(rows)}
        for k, r in enumerate(rows):
            t = self.timepoints[r]
            if self.row_succ[r] is not None and t + 1 < t1 + W:
                nxt = self.rows(t + 1)
                M[k, loc[int(nxt[0])]:loc[int(nxt[-1])] + 1] = 1.0
                for s in self.row_succ[r]:
                    A[k, loc[s]] = 1.0
            p = self.col_pred[r]
            if p >= 0 and t - 1 >= t1:
                prv = self.rows(t - 1)
                M[loc[int(prv[0])]:loc[int(prv[-1])] + 1, k] = 1.0
                A[loc[int(p)], k] = 1.0
        feat = self.wrfeatures(rows)
        if augmenter is not None:
            feat = augmenter(feat)
        coords = np.concatenate((feat.timepoints[:, None].astype(np.float32), feat.coords.astype(np.float32)), axis=1)
        return dict(coords=torch.from_numpy(coords), features=torch.from_numpy(feat.features_stacked.astype(np.float32)),
                    timepoints=torch.from_numpy(feat.timepoints.astype(np.int64)),
                    A=torch.from_numpy(A), M=torch.from_numpy(M), n=n)


def make_augmenter(level=2):
    from trackastra.data import wrfeat
    if level <= 0:
        return None
    augs = [wrfeat.WRRandomFlip(p=0.5),
            wrfeat.WRRandomAffine(p=0.8, degrees=180, scale=(0.8, 1.25), shear=(0.1, 0.1))]
    if level >= 2:
        augs += [wrfeat.WRRandomBrightness(p=0.8), wrfeat.WRRandomOffset(p=0.8, offset=(-1, 1))]
    return wrfeat.WRAugmentationPipeline(augs)


# ------------------------------------------------------------------ loss (mirrors trackastra scripts/train.py, plus our mask)
def window_loss(transformer, w, device, causal_norm="quiet_softmax", div_upweight=3.0, eps=1e-6, use_amp=False):
    """Forward in fp16 autocast (if use_amp); the loss itself is always computed in fp32 with autocast disabled,
    because torch refuses binary_cross_entropy under CUDA autocast."""
    import torch
    import torch.nn.functional as Fnn
    from trackastra.utils import blockwise_causal_norm, blockwise_sum

    coords = w["coords"][None].to(device); feats = w["features"][None].to(device)
    tp = w["timepoints"].to(device); A = w["A"].to(device); M = w["M"].to(device)
    with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp and device.type == "cuda"):
        A_pred = transformer(coords, feats, padding_mask=None)[0]
    with torch.autocast(device.type, enabled=False):
        A_pred = A_pred.float().clamp(-1e4, 1e4)
        loss = Fnn.binary_cross_entropy_with_logits(A_pred, A, reduction="none")
        if causal_norm != "none":
            A_soft = blockwise_causal_norm(A_pred, tp, mode=causal_norm).clamp(eps, 1 - eps)
            loss = 0.01 * loss + Fnn.binary_cross_entropy(A_soft, A, reduction="none")
    with torch.no_grad():
        bs = A * (blockwise_sum(A, tp, dim=-1) + blockwise_sum(A, tp, dim=-2))
        weight = 1 + 1.0 * (bs == 2).float() + div_upweight * (bs > 2).float()
        # accuracy on supervised rows: argmax over the next-frame block hits a target
        dt = tp[None, :] - tp[:, None]
        row_sup = (A * (dt == 1)).sum(1) > 0            # rows with a known successor
        acc_n = acc_hit = 0
        if row_sup.any() and causal_norm != "none":
            sc = A_soft.masked_fill(dt != 1, -1.0)
            j = sc[row_sup].argmax(1)
            acc_hit = int(A[row_sup][torch.arange(int(row_sup.sum()), device=device), j].sum().item())
            acc_n = int(row_sup.sum().item())
    loss = (loss * weight * M).sum() / (M.sum() + eps)
    return loss, dict(acc_hit=acc_hit, acc_n=acc_n, n_sup=float(M.sum().item()))


# ------------------------------------------------------------------ training loop
def finetune(transformer, train_caches, val_caches, out_dir: Path, *, W=4, steps=6000, lr=3e-5, warmup=300,
             accum=8, augment=2, max_tokens=3000, time_budget_sec=4 * 3600, eval_every=1000, eval_windows=150,
             causal_norm="quiet_softmax", div_upweight=3.0, seed=0, log=print):
    """Fine-tune `transformer` in place. Saves pretrained.pt / best_val.pt / last.pt to out_dir. Returns history.
    (The final choice between them is made with the official-style CV, see select_and_save.)"""
    import torch
    rng = np.random.RandomState(seed); torch.manual_seed(seed)
    device = next(transformer.parameters()).device
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    opt = torch.optim.AdamW(transformer.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / max(1, steps)))))
    aug = make_augmenter(augment)
    tr_starts = [(c, c.window_starts(W)) for c in train_caches]
    tr_starts = [(c, s) for c, s in tr_starts if s]
    va_items = []
    for c in val_caches:
        for t1 in c.window_starts(W):
            va_items.append((c, t1))
    rng.shuffle(va_items); va_items = va_items[:eval_windows]
    log(f"train clips with supervision: {len(tr_starts)}/{len(train_caches)} | "
        f"supervised rows {sum(c.n_sup_rows for c in train_caches)} cols {sum(c.n_sup_cols for c in train_caches)} | "
        f"val windows {len(va_items)} | steps {steps} x accum {accum} | device {device}")

    def evaluate():
        transformer.eval(); tot = n = hit = nn_ = 0.0
        with torch.no_grad():
            for c, t1 in va_items:
                w = c.make_window(t1, W, None, max_tokens)
                l, st = window_loss(transformer, w, device, causal_norm, div_upweight, use_amp=use_amp)
                tot += float(l); n += 1; hit += st["acc_hit"]; nn_ += st["acc_n"]
        transformer.train()
        return tot / max(1, n), hit / max(1, nn_)

    hist = []
    v0 = evaluate() if va_items else (float("nan"), float("nan"))
    log(f"  step 0 (pretrained): val loss {v0[0]:.4f} | val link acc {v0[1]:.4f}")
    hist.append(dict(step=0, val_loss=v0[0], val_acc=v0[1], train_loss=None))
    best = float("inf")                       # first evaluated checkpoint always becomes best_val.pt
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(transformer.state_dict(), out_dir / "pretrained.pt")
    transformer.train(); t0 = time.time(); run_loss, run_n, run_hit, run_sup = 0.0, 0, 0, 0
    step = 0
    while step < steps:
        opt.zero_grad(set_to_none=True)
        for _ in range(accum):
            c, starts = tr_starts[rng.randint(len(tr_starts))]      # clip-uniform sampling (dense & sparse embryos equal)
            t1 = starts[rng.randint(len(starts))]
            w = c.make_window(t1, W, aug, max_tokens)
            if w["M"].sum() == 0:
                continue
            l, st = window_loss(transformer, w, device, causal_norm, div_upweight, use_amp=use_amp)
            if not torch.isfinite(l):
                log("    non-finite loss, skipping window"); continue
            scaler.scale(l / accum).backward()
            run_loss += float(l); run_n += 1; run_hit += st["acc_hit"]; run_sup += st["acc_n"]
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(transformer.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step(); step += 1
        if step % 100 == 0:
            log(f"  step {step}/{steps} lr {sched.get_last_lr()[0]:.2e} train loss {run_loss/max(1,run_n):.4f} "
                f"train link acc {run_hit/max(1,run_sup):.4f} ({time.time()-t0:.0f}s)")
        if step % eval_every == 0 or step == steps or time.time() - t0 > time_budget_sec:
            vl, va = evaluate() if va_items else (float("nan"), float("nan"))
            hist.append(dict(step=step, val_loss=vl, val_acc=va, train_loss=run_loss / max(1, run_n)))
            log(f"  == step {step}: val loss {vl:.4f} | val link acc {va:.4f} | train loss {run_loss/max(1,run_n):.4f}")
            run_loss, run_n, run_hit, run_sup = 0.0, 0, 0, 0
            torch.save(transformer.state_dict(), out_dir / "last.pt")
            if not va_items or vl < best:
                best = vl; torch.save(transformer.state_dict(), out_dir / "best_val.pt")
                log(f"     new best val loss -> saved {out_dir/'best_val.pt'}")
            if time.time() - t0 > time_budget_sec:
                log(f"  time budget reached at step {step}"); break
    (out_dir / "history.json").write_text(json.dumps(hist, indent=1))
    return hist


def save_model_folder(transformer, state_dict, src_model_dir: Path, out_dir: Path, note: str = ""):
    """Write a folder loadable by Trackastra.from_folder (config.yaml + train_config.yaml + model.pt)."""
    import torch, yaml
    out_dir.mkdir(parents=True, exist_ok=True)
    yaml.safe_dump(transformer.config, open(out_dir / "config.yaml", "w"))
    ta = yaml.safe_load(open(Path(src_model_dir) / "train_config.yaml"))
    ta["finetuned_on"] = "biohub-cell-tracking-during-development " + note
    yaml.safe_dump(ta, open(out_dir / "train_config.yaml", "w"))
    torch.save(state_dict, out_dir / "model.pt")


def select_and_save(transformer, work_dir: Path, holdout_caches, cfg, gt_dir: Path, src_model_dir: Path, out_dir: Path, log=print):
    """Score pretrained / best_val / last on the hold-out clips with the official-style metric, keep the winner."""
    import torch
    results = {}
    if not (work_dir / "pretrained.pt").exists():          # training never started -> at least evaluate/save the start weights
        work_dir.mkdir(parents=True, exist_ok=True)
        torch.save(transformer.state_dict(), work_dir / "pretrained.pt")
    for tag in ("pretrained", "best_val", "last"):
        p = work_dir / f"{tag}.pt"
        if not p.exists():
            continue
        transformer.load_state_dict(torch.load(p, map_location=next(transformer.parameters()).device, weights_only=True))
        _, ma = cv_from_cache(holdout_caches, transformer, cfg, gt_dir, label=tag, log=log)
        results[tag] = ma["score"]
    winner = max(results, key=results.get)
    sd = torch.load(work_dir / f"{winner}.pt", map_location="cpu", weights_only=True)
    save_model_folder(transformer, sd, src_model_dir, out_dir, note=f"selected={winner} holdout_cv={results}")
    (out_dir / "SELECTED.txt").write_text(json.dumps(dict(selected=winner, holdout_cv=results), indent=1))
    log(f"hold-out CV scores: {results} -> selected '{winner}' -> {out_dir}")
    if winner == "pretrained":
        log("!! fine-tuning did NOT beat the pretrained model on the hold-out clips; ctc_ft = pretrained weights")
    return winner, results


# ------------------------------------------------------------------ CV from cache (no re-detection needed)
def link_from_cache(cache: ClipCache, transformer, cfg, batch_size=4, log=print):
    """Reproduce link_trackastra() from cached features: predict_windows -> greedy -> division pruning."""
    import torch
    from trackastra.data import build_windows
    from trackastra.model.predict import predict_windows
    from trackastra.tracking import build_graph, track_greedy

    feats = cache.frame_wrfeatures()
    windows = build_windows(feats, window_size=transformer.config["window"], progbar_class=lambda x, **k: x, as_torch=True)
    transformer.eval()
    with torch.no_grad():
        pred = predict_windows(windows=windows, features=feats, model=transformer, edge_threshold=0.05,
                               spatial_dim=3, batch_size=batch_size, progbar_class=lambda x, **k: x)
    G = build_graph(nodes=pred["nodes"], weights=pred["weights"], use_distance=False,
                    max_distance=cfg["trackastra_max_distance"], max_neighbors=10, delta_t=1)
    graph = track_greedy(G) if cfg.get("trackastra_mode", "greedy") == "greedy" else track_greedy(G, allow_divisions=False)
    key2id, nodes = {}, []
    for r in range(len(cache.labels)):
        t, lab = int(cache.timepoints[r]), int(cache.labels[r])
        key2id[(t, lab)] = r + 1
        z, y, x = cache.det_zyx[r]
        nodes.append((t, int(z), int(y), int(x), r + 1))
    edges, weights = [], []
    for u, v, d in graph.edges(data=True):
        nu, nv = graph.nodes[u], graph.nodes[v]
        ku, kv = (int(nu["time"]), int(nu["label"])), (int(nv["time"]), int(nv["label"]))
        if ku in key2id and kv in key2id:
            edges.append((key2id[ku], key2id[kv])); weights.append(float(d.get("weight", 1.0)))
    edges = prune_divisions(nodes, edges, weights, cfg, log=lambda *a: None)
    return nodes, edges


def cv_from_cache(caches, transformer, cfg, gt_dir: Path, label="", log=print):
    rows = []
    for c in caches:
        gt = read_geff(gt_dir / (c.name + ".geff"))
        nodes, edges = link_from_cache(c, transformer, cfg)
        sc = score_official(nodes, edges, gt); sc.update(config=label, sample=c.name)
        rows.append(sc)
        log(f"    [{label}] {c.name}: score={sc['score']:.3f} (tp={sc['edge_tp']} fp={sc['edge_fp']} fn={sc['edge_fn']}) "
            f"node_recall={sc['node_recall']:.3f} n_pred/est={sc['n_pred']}/{sc['est']}")
    ma = micro_average(rows); ma["config"] = label
    log(f"  ==> [{label}] micro score={ma['score']:.4f} adjJ={ma['adj_edge_J']:.4f} divJ={ma['div_J']:.3f}")
    return rows, ma


In [ ]:
# ------------------------------------------------------------------ 4. load pretrained model, choose clips
from trackastra.model import Trackastra
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = Trackastra.from_folder(Path(MODEL_DIR), device=dev)
transformer = model.transformer
print("start weights:", MODEL_DIR, "| pretrained folder:", PRETRAINED_DIR, "| device", dev,
      "| causal_norm", transformer.config["causal_norm"], "| window", transformer.config["window"])
CFG.update(DETECT_CONFIG)

train_dir = COMP_DIR / "train"
all_zarrs = sorted(p for p in train_dir.iterdir() if p.name.endswith(".zarr") and (train_dir / (p.name[:-5] + ".geff")).exists())
step = max(1, len(all_zarrs) // max(1, N_HOLDOUT))
HOLDOUT = all_zarrs[1::step][:N_HOLDOUT]                    # identical rule to the inference notebook's RUN_CV
rest = [p for p in all_zarrs if p not in set(HOLDOUT)]
# interleave so that every embryo appears early (matters if the feature budget stops extraction early)
by_embryo = {}
for p in rest:
    by_embryo.setdefault(p.name.split("_")[0], []).append(p)
TRAIN = []
while any(by_embryo.values()):
    for k in list(by_embryo):
        if by_embryo[k]:
            TRAIN.append(by_embryo[k].pop(0))
if MAX_TRAIN_CLIPS:
    TRAIN = TRAIN[:MAX_TRAIN_CLIPS]
print(f"{len(all_zarrs)} train clips -> hold-out {len(HOLDOUT)}: {[p.name[:-5] for p in HOLDOUT]}")
print(f"fine-tune on up to {len(TRAIN)} clips, first: {[p.name[:-5] for p in TRAIN[:6]]} ...")


In [ ]:
# ------------------------------------------------------------------ 5. feature cache (re-use attached caches, extract the rest)
CACHE_OUT = OUT_DIR / "feat_cache"; CACHE_OUT.mkdir(exist_ok=True)
existing = {}
for d in CACHE_DIRS + [CACHE_OUT]:
    for p in sorted(d.glob("*.npz")):
        existing.setdefault(p.name[:-4], p)
print(f"{len(existing)} cached clips found in attached inputs")

def _cache_path(zp):
    name = zp.name[:-5]
    if name in existing:
        return existing[name]
    if time.time() - T_START > FEATURE_BUDGET_SEC:
        return None
    out = CACHE_OUT / (name + ".npz")
    extract_clip_cache(zp, train_dir / (name + ".geff"), CFG, out, n_workers=FEATURE_WORKERS)
    existing[name] = out
    return out

t0 = time.time(); n_new = 0
hold_npz = []
for zp in HOLDOUT:                                  # hold-out first: they are needed for the final selection
    p = _cache_path(zp); hold_npz.append(p)
train_npz = []
for k, zp in enumerate(TRAIN):
    p = _cache_path(zp)
    if p is None:
        print(f"!! feature budget reached: {k}/{len(TRAIN)} train clips cached"); break
    train_npz.append(p)
    if k % 10 == 9:
        print(f"    ... {k+1}/{len(TRAIN)} train clips, elapsed {time.time()-T_START:.0f}s")
hold_npz = [p for p in hold_npz if p is not None]
print(f"features ready: {len(hold_npz)} hold-out + {len(train_npz)} train clips ({time.time()-t0:.0f}s)")

hold_caches = [ClipCache(p) for p in hold_npz]
train_caches = [ClipCache(p) for p in train_npz]
import pandas as pd
tab = pd.DataFrame([dict(clip=c.name, frames=c.n_frames, detections=len(c.labels), radius=c.radius,
                         gt_nodes=len(c.gt_ids), gt_matched=int((c.gt_id >= 0).sum()), gt_edges=len(c.gt_edges),
                         sup_rows=c.n_sup_rows, sup_cols=c.n_sup_cols, split="holdout" if c in hold_caches else "train")
                   for c in hold_caches + train_caches])
display(tab.groupby("split")[["detections", "gt_nodes", "gt_matched", "gt_edges", "sup_rows", "sup_cols"]].sum())
display(tab.head(15))
tab.to_csv(OUT_DIR / "clips_table.csv", index=False)


In [ ]:
# ------------------------------------------------------------------ 6. fine-tune
WORK = OUT_DIR / "ft_work"
remaining = max(600, min(TRAIN_BUDGET_SEC, 11.0 * 3600 - (time.time() - T_START) - 40 * 60))   # leave ~40 min for CV + save
print(f"training budget: {remaining/3600:.2f} h")
import traceback
try:
    hist = finetune(transformer, train_caches, hold_caches, WORK, W=transformer.config["window"], steps=TRAIN_STEPS, lr=LR,
                    warmup=WARMUP_STEPS, accum=ACCUM, augment=AUGMENT_LEVEL, max_tokens=MAX_TOKENS, time_budget_sec=remaining,
                    eval_every=EVAL_EVERY, eval_windows=150, causal_norm=transformer.config["causal_norm"], seed=SEED)
    display(pd.DataFrame(hist))
except Exception:
    # never let a training crash lose the 2 h feature cache: the notebook must finish so the Output is kept
    print("!! TRAINING FAILED — the feat_cache is still written to the Output; attach it next time to skip extraction")
    traceback.print_exc()


In [ ]:
# ------------------------------------------------------------------ 7. select by official-style CV on the hold-out clips, save models/ctc_ft
FT_DIR = OUT_DIR / "models" / "ctc_ft"
winner, cv_scores = select_and_save(transformer, WORK, hold_caches, CFG, train_dir, PRETRAINED_DIR, FT_DIR)
print("\nhold-out micro scores:", cv_scores)
print("selected:", winner, "->", sorted(p.name for p in FT_DIR.iterdir()))
for p in ("pretrained.pt", "last.pt", "best_val.pt"):          # keep output small (each is 110 MB)
    (WORK / p).unlink(missing_ok=True)
print(f"total {time.time()-T_START:.0f}s. Next: Save Version -> Save & Run All; then attach this notebook's Output to the "
      f"inference notebook (it prefers models/ctc_ft). Re-running this notebook with its own Output attached skips feature extraction.")
